# 資料處理與重新編碼規範 (Recoding Rules Documentation)

本文件記錄本專案自原始 YRBSS 資料集至推論統計模型建置研究所實施之完整重新編碼規則（Recoding Rules），以確保資料清洗、變項轉換與統計假設之嚴謹性。

---

## 一、 樣本清洗與缺失值處理規則 (Data Cleaning & Listwise Deletion)
* **操作定義**：本專案針對 8 大核心變數（香菸、酒精、大麻及五項硬性毒品）發動**完全填答個案刪除法（Listwise Deletion）**。
* **重新編碼規則**：剔除任何在上述任一變數中含有 `NaN`（未填答、漏填或拒答）的觀測值。
* **統計效果**：自原始資料集中篩選出完全填答之有效樣本共計 $N = 12,093$ 筆，作為後續所有微觀與全域推論統計之「黃金分析樣本池」。

## 二、 自變數（門檻物質）二元化與 8 大群體分流規則
為深入探討多重物質並用背景，本研究將香菸、酒精、大麻之原始 1 至 6 級分進行**二元涉入化（Binarization）**切分：

1. **原始級分基準（依據 YRBSS 規範）**：
   * **級分 1.0**：0 次（代表該物質未涉入）
   * **級分 2.0 ~ 6.0**：1 次以上（代表該物質有涉入，包含 1-2次、3-9次、10-19次、20-39次、40次以上）
2. **二元編碼公式**：
   * `Cig_User = 1` if `CurrentCigaretteUse >= 2.0` else `0`
   * `Alc_User = 1` if `EverAlcoholUse >= 2.0` else `0`
   * `Mj_User  = 1` if `CurrentMarijuaUse >= 2.0` else `0`
3. **8 大互斥物質涉入群體（Substance Group）分類矩陣**：

| 群體代號 | 群體中文標準名稱 | 香菸涉入 (`Cig_User`) | 酒精涉入 (`Alc_User`) | 大麻涉入 (`Mj_User`) |
| :---: | :--- | :---: | :---: | :---: |
| **G1** | 完全清白組 (None) | 0 | 0 | 0 |
| **G2** | 單獨吸菸組 (Cig Only) | 1 | 0 | 0 |
| **G3** | 單獨飲酒組 (Alc Only) | 0 | 1 | 0 |
| **G4** | 單獨大麻組 (Mj Only) | 0 | 0 | 1 |
| **G5** | 菸酒共用組 (Cig + Alc) | 1 | 1 | 0 |
| **G6** | 菸大麻共用組 (Cig + Mj) | 1 | 0 | 1 |
| **G7** | 酒大麻共用組 (Alc + Mj) | 0 | 1 | 1 |
| **G8** | 三者全面涉入組 (All Three) | 1 | 1 | 1 |

## 三、 因變數（硬性毒品）基準平移規則 (Score Centering)
在原始數據中，數值 `1.0` 代表「0次（從未吸食）」。為使線性迴歸（OLS）模型之截距項（Intercept）具備實質之科學可讀性（即當自變數皆為 0 時，預測值亦對齊為 0），本研究對所有硬性毒品進行**減 1 平移轉換**。

* **數學轉換公式**：

$$\text{Drug\_Score} = \text{EverDrugUse} - 1$$

* **重新編碼映射對照表**：

| 原始填答級分 (YRBSS) | 重新編碼後得分 (`Score`) | 實際行為代表意義 |
| :---: | :---: | :--- |
| **1.0** | **0** | **從未吸食 (0 times)** |
| 2.0 | 1 | 嘗試過 1~2 次 |
| 3.0 | 2 | 嘗試過 3~9 次 |
| 4.0 | 3 | 嘗試過 10~19 次 |
| 5.0 | 4 | 嘗試過 20~39 次 |
| 6.0 | 5 | 嘗試過 40 次以上 |

* **涵蓋目標變項**：`Cocaine_Score`, `Heroin_Score`, `Inhalant_Score`, `Meth_Score`, `Ecstasy_Score`。

## 四、 綜合硬性毒品指標建置規則 (Composite Hard Drug Index)
為了評估青少年對於硬性毒品整體的「最高涉入程度」，而非單一毒品行為，本研究建置了綜合應變數指標。

* **建置規則**：提取五種硬性毒品原始填答級分的**最大值（Maximum Engagement）**，再進行減 1 平移。
* **程式碼實作管線**：
  ```python
  hard_drug_columns = ['EverCocaineUse', 'EverHeroinUse', 'EverInhalantUse', 'EverMethamphetamineUse', 'EverEcstasyUse']
  df_processed['AnyHardDrug_Int'] = df_processed[hard_drug_columns].max(axis=1).astype(int)
  df_processed['AnyHardDrug_Score'] = df_processed['AnyHardDrug_Int'] - 1

# Data Processing and Recoding Specifications (Recoding Rules Documentation)

This document details the complete recoding rules implemented in this project to transform the raw YRBSS dataset into a structure suitable for predictive and inferential statistical modeling. These rules ensure statistical rigor, control for confounding variables, and fulfill the mathematical assumptions of linear regression models.

---

## I. Sample Cleaning and Missing Value Treatment (Data Cleaning & Listwise Deletion)
* **Operational Definition**: This project utilizes **Listwise Deletion** across the 8 core substance variables (cigarettes, alcohol, marijuana, and five hard drugs).
* **Recoding Rule**: Any observation/record containing a `NaN` (blank, missing, or "Refused to Answer") value in *any* of the 8 core variables is completely dropped from the active dataset.
* **Statistical Impact**: A fully-answered "Golden Analysis Sample Pool" of $N = 12,093$ valid records was extracted from the raw dataset, serving as the consistent baseline for all subsequent micro-level and global inferential analyses.

## II. Binarization of Independent Variables & 8 Mutually Exclusive Groups Stratification
To isolate the net effect of marijuana under distinct substance environments, the original 1.0–6.0 scales for current cigarette use, lifetime alcohol use, and current marijuana use were processed using a **Binarization** rule:

1. **YRBSS Baseline Scales**:
   * **Score 1.0**: 0 times (indicates non-engagement/abstinence for that substance).
   * **Score 2.0 to 6.0**: 1 or more times (indicates engagement, spanning 1-2 times, 3-9 times, 10-19 times, 20-39 times, and 40+ times).
2. **Binarization Logic**:
   * `Cig_User = 1` if `CurrentCigaretteUse >= 2.0` else `0`
   * `Alc_User = 1` if `EverAlcoholUse >= 2.0` else `0`
   * `Mj_User  = 1` if `CurrentMarijuaUse >= 2.0` else `0`
3. **8 Mutually Exclusive Substance Engagement Groups Classification Matrix**:

| Group Code | Academic Standard Group Name | Tobacco Engagement (`Cig_User`) | Alcohol Engagement (`Alc_User`) | Marijuana Engagement (`Mj_User`) |
| :---: | :--- | :---: | :---: | :---: |
| **G1** | None/Abstinent Group | 0 | 0 | 0 |
| **G2** | Cigarette Only Group | 1 | 0 | 0 |
| **G3** | Alcohol Only Group | 0 | 1 | 0 |
| **G4** | Marijuana Only Group | 0 | 0 | 1 |
| **G5** | Cigarette + Alcohol Group | 1 | 1 | 0 |
| **G6** | Cigarette + Marijuana Group | 1 | 0 | 1 |
| **G7** | Alcohol + Marijuana Group | 0 | 1 | 1 |
| **G8** | All Three/Polysubstance Group | 1 | 1 | 1 |

## III. Score Centering and Baseline Translation of Dependent Variables (Hard Drugs)
In the raw YRBSS configuration, a value of `1.0` represents "0 times (never used)." To make the intercept ($\beta_0$) of Ordinary Least Squares (OLS) regression models mathematically and academically intuitive (ensuring that the predicted score equals 0 when the independent behavior is absent), a **minus-1 baseline translation** was applied to all hard drug scores.

* **Mathematical Formula**:

$$\text{Drug\_Score} = \text{EverDrugUse} - 1$$

* **Recoding and Mapping Reference Table**:

| Raw YRBSS Score | Recoded Outcome Score (`Score`) | Behavioral Interpretation |
| :---: | :---: | :--- |
| **1.0** | **0** | **Never Used (0 times)** |
| 2.0 | 1 | Experimented 1–2 times |
| 3.0 | 2 | Experimented 3–9 times |
| 4.0 | 3 | Experimented 10–19 times |
| 5.0 | 4 | Experimented 20–39 times |
| 6.0 | 5 | Experimented 40 or more times |

* **Target Variables Covered**: `Cocaine_Score`, `Heroin_Score`, `Inhalant_Score`, `Meth_Score`, `Ecstasy_Score`.

## IV. Construction of the Composite Hard Drug Index (Composite Hard Drug Index)
To assess an adolescent’s overall "maximum severity of involvement" across any hard drug behavior rather than a single category, a composite dependent variable index was constructed.

* **Rule**: Extract the **Maximum Engagement** value across the five hard drug raw scales for each individual, and then apply the minus-1 score centering.
* **Code Implementation Pipeline**:
  ```python
  hard_drug_columns = ['EverCocaineUse', 'EverHeroinUse', 'EverInhalantUse', 'EverMethamphetamineUse', 'EverEcstasyUse']
  df_processed['AnyHardDrug_Int'] = df_processed[hard_drug_columns].max(axis=1).astype(int)
  df_processed['AnyHardDrug_Score'] = df_processed['AnyHardDrug_Int'] - 1